[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yildiztu/Code-101-Problems-Logistics-SCM/blob/main/Part%20II%20-%20The%20End-to-End%20Supply%20Chain%20%28Problems%2047-101%29/E.%20Integrated%2C%20Resilient%2C%20%26%20Modern%20Supply%20Chains%20%28The%20Frontier%29/096.%20The%20Resilient%20Network%20Design%20%28Robust%20Optimization%29/P96-Tier-9_executed.ipynb)

---


# 96. Resilient Network Design with QAOA
## Tier 9: A Quantum Approximation Optimization Algorithm

### Purpose and scope

This notebook demonstrates the main stages of a QAOA-style approach to a binary facility-selection problem: formulate a quadratic unconstrained binary optimization (QUBO) objective, prepare a statevector, alternate cost and mixer operations, sample binary decisions, and tune circuit parameters with a gradient-free random-perturbation procedure.

The supplied example has eight candidate facilities and four demand zones. Each facility is represented by one binary decision variable. The objective combines facility fixed costs with a pairwise distance-based diversification reward. The demand-zone data are retained for context and visualization, but the stated QUBO does not encode demand coverage, facility capacity, disruption scenarios, or service assignments. Consequently, this is a stylized facility-selection objective, not a complete resilient network-design model. The penalty weights and claims of capacity or coverage enforcement are not part of the formulation.

The implementation is a **classical statevector simulation** of QAOA. It does not run on quantum hardware and does not establish quantum advantage, speedup, or an accuracy percentage. An \(n\)-qubit statevector contains \(2^n\) complex amplitudes, so this simulation has exponential memory requirements in \(n\). The example is small enough to illustrate the circuit mechanics, not to demonstrate a computational advantage.

### Model assumptions and objective

Let \(F=\{0,\ldots,n-1\}\) index facilities, and let \(x_i\in\{0,1\}\) indicate whether facility \(i\) is selected. The objective encoded by the notebook is

$$
C(x)=c+\sum_{i\in F} h_i x_i+\sum_{\substack{i,j\in F\\i<j}}J_{ij}x_i x_j,
\qquad
h_i=\text{fixed cost of facility }i,
\qquad
J_{ij}=-0.1\,\frac{d_{ij}}{1000},
$$

where \(c=0\), and \(d_{ij}\) is the Euclidean distance between the two facility coordinates. The pairwise term is a stipulated diversification reward: selecting both facilities lowers the objective by an amount that increases with their separation. Coordinates are treated as given coordinate units, and the stated coefficient defines the scale of this reward. The resulting objective is a composite score; its value should not be interpreted as a dollar total. No constraint requires any facility to be selected, so the all-zero decision is feasible under this model.

In the code, \(h_i\) is stored in `linear_terms`, \(J_{ij}\) in `quadratic_terms`, and \(c\) in `constant_term`. The full binary decision vector is the QUBO state. For a binary vector \(x\), its objective is evaluated by `calculate_objective`.

### QAOA state and parameter search

QAOA represents candidate decisions as computational-basis states. It starts from the uniform superposition and applies a cost operation, which adds objective-dependent phases, followed by a mixer operation, which changes amplitudes among bit strings. Repeating these operations for \(p\) layers gives

$$
|\psi(\boldsymbol{\gamma},\boldsymbol{\beta})\rangle
=
\prod_{\ell=0}^{p-1}
\left(e^{-i\beta_\ell H_M}e^{-i\gamma_\ell H_C}\right)|+\rangle^{\otimes n},
\qquad
H_C|x\rangle=C(x)|x\rangle,
\qquad
H_M=\sum_{i=0}^{n-1}X_i.
$$

Here, \(|+\rangle^{\otimes n}\) assigns equal initial amplitude to every binary decision, \(H_C\) is diagonal with the QUBO objective as its basis-state energy, and \(X_i\) is the Pauli-\(X\) operator on qubit \(i\). The code implements the cost phase directly from the linear and pairwise QUBO terms. Its mixer uses \(e^{-i\beta X_i/2}\) rotations, a parameter convention in which the factor of one-half is included in each rotation. Each layer has one cost parameter \(\gamma_\ell\) and one mixer parameter \(\beta_\ell\), for \(2p\) parameters in total.

The parameter search estimates the mean objective from measurement samples. It perturbs the current parameters randomly, retains the best parameter vector observed by that sampled estimate, and then samples the resulting state to report the best measured decision. Sampling makes these estimates stochastic. The statevector update is exponential in the number of qubits; in this implementation, applying the gates directly costs on the order of \(p n^2 2^n\) operations per state evolution. This notebook follows this QAOA and parameter-search sequence; its results describe this simulation only.

In [ ]:
from dataclasses import dataclass, field
from typing import Dict, List, Optional, Any
import random

import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns

plt.style.use("seaborn-v0_8")
sns.set_palette("husl")

## Data structures

The data classes keep the problem inputs, QUBO coefficients, circuit state, and reported results explicit. `quantum_amplitude`, `entanglement_partners`, `quantum_priority`, and `superposition_states` are retained as illustrative data fields, but they do not enter the QUBO or the statevector calculation. In particular, these metadata fields do not create physical entanglement or encode demand priorities in the model.

In [ ]:
@dataclass
class Facility:
    """Candidate facility and its selection cost."""

    id: int
    x: float
    y: float
    capacity: int
    fixed_cost: float
    operational_status: bool = True
    quantum_amplitude: complex = 1.0 + 0.0j
    entanglement_partners: Optional[List[int]] = None


@dataclass
class DemandZone:
    """Demand-zone data retained for context and visualization."""

    id: int
    x: float
    y: float
    base_demand: float
    quantum_priority: float = 1.0
    superposition_states: Optional[List[bool]] = None


@dataclass
class QuantumCircuit:
    """Statevector and parameters for the QAOA circuit."""

    num_qubits: int
    depth: int
    parameters: List[float]
    mixer_hamiltonian: str = "X"
    cost_hamiltonian: str = "QUBO"
    quantum_state: Optional[np.ndarray] = None


@dataclass
class QUBOFormulation:
    """Quadratic binary objective in linear and pairwise coefficient form."""

    linear_terms: np.ndarray
    quadratic_terms: np.ndarray
    constant_term: float
    num_variables: int


@dataclass
class QuantumResult:
    """Outputs produced by the statevector QAOA simulation."""

    best_solution: List[int]
    best_energy: float
    convergence_history: List[float]
    parameter_history: List[List[float]]
    circuit_depth: int
    quantum_metrics: Dict[str, float]
    classical_comparison: Dict[str, float] = field(default_factory=dict)
    sampled_energies: List[float] = field(default_factory=list)

## QUBO formulation and QAOA simulation

The optimizer below keeps the solution sequence visible: it builds the QUBO, initializes a uniform state, applies the cost and mixer operations layer by layer, samples candidate bit strings, and searches over circuit parameters. A basis-state index stores its binary decision in its bits: bit \(i\) corresponds to facility \(i\). This convention connects the statevector operations directly to the decision vector used by the objective function.

In [ ]:
class QuantumNetworkOptimizer:
    """Statevector QAOA simulation for the stated facility-selection QUBO."""

    def __init__(
        self,
        facilities: List[Facility],
        demand_zones: List[DemandZone],
    ) -> None:
        if not facilities:
            raise ValueError("At least one facility is required.")

        self.facilities = facilities
        self.demand_zones = demand_zones
        self.num_facilities = len(facilities)
        self.num_qubits = self.num_facilities
        self.convergence_history: List[float] = []

    def formulate_qubo(self) -> QUBOFormulation:
        """Build the fixed-cost and distance-reward objective."""
        n = self.num_facilities
        linear_terms = np.array(
            [facility.fixed_cost for facility in self.facilities],
            dtype=float,
        )
        quadratic_terms = np.zeros((n, n), dtype=float)

        for i in range(n):
            for j in range(i + 1, n):
                dx = self.facilities[i].x - self.facilities[j].x
                dy = self.facilities[i].y - self.facilities[j].y
                distance = np.sqrt(dx**2 + dy**2)
                reward = -0.1 * distance / 1000
                quadratic_terms[i, j] = reward
                quadratic_terms[j, i] = reward

        return QUBOFormulation(
            linear_terms=linear_terms,
            quadratic_terms=quadratic_terms,
            constant_term=0.0,
            num_variables=n,
        )

    def initialize_quantum_circuit(self, depth: int = 3) -> QuantumCircuit:
        """Create a uniform initial state and one parameter pair per layer."""
        if depth < 1:
            raise ValueError("Circuit depth must be at least one.")

        parameters = [random.uniform(0.0, 2.0 * np.pi) for _ in range(2 * depth)]
        state_size = 2**self.num_qubits
        quantum_state = np.ones(state_size, dtype=complex) / np.sqrt(state_size)

        return QuantumCircuit(
            num_qubits=self.num_qubits,
            depth=depth,
            parameters=parameters,
            quantum_state=quantum_state,
        )

    def apply_cost_hamiltonian(
        self,
        circuit: QuantumCircuit,
        qubo: QUBOFormulation,
        gamma: float,
    ) -> None:
        """Apply the QUBO-dependent phase to each computational basis state."""
        if circuit.quantum_state is None:
            raise ValueError("The circuit state has not been initialized.")

        for i in range(self.num_qubits):
            angle = -gamma * qubo.linear_terms[i]
            circuit.quantum_state = self.apply_z_rotation(
                circuit.quantum_state, i, angle
            )

        for i in range(self.num_qubits):
            for j in range(i + 1, self.num_qubits):
                coefficient = qubo.quadratic_terms[i, j]
                if not np.isclose(coefficient, 0.0):
                    angle = -gamma * coefficient
                    circuit.quantum_state = self.apply_zz_interaction(
                        circuit.quantum_state, i, j, angle
                    )

    def apply_mixer_hamiltonian(
        self,
        circuit: QuantumCircuit,
        beta: float,
    ) -> None:
        """Apply the product of single-qubit X-mixer rotations."""
        if circuit.quantum_state is None:
            raise ValueError("The circuit state has not been initialized.")

        for qubit in range(self.num_qubits):
            circuit.quantum_state = self.apply_x_rotation(
                circuit.quantum_state, qubit, beta
            )

    def apply_z_rotation(
        self,
        state: np.ndarray,
        qubit: int,
        angle: float,
    ) -> np.ndarray:
        """Apply a diagonal phase to states whose selected bit is one."""
        new_state = state.copy()
        for basis_index in range(len(state)):
            if (basis_index >> qubit) & 1:
                new_state[basis_index] *= np.exp(1j * angle)
        return new_state

    def apply_x_rotation(
        self,
        state: np.ndarray,
        qubit: int,
        angle: float,
    ) -> np.ndarray:
        """Apply exp(-i angle X / 2) to one qubit of the statevector."""
        new_state = np.zeros_like(state)
        cosine = np.cos(angle / 2.0)
        negative_i_sine = -1j * np.sin(angle / 2.0)

        for basis_index, amplitude in enumerate(state):
            flipped_index = basis_index ^ (1 << qubit)
            new_state[basis_index] += cosine * amplitude
            new_state[flipped_index] += negative_i_sine * amplitude

        return new_state

    def apply_zz_interaction(
        self,
        state: np.ndarray,
        qubit1: int,
        qubit2: int,
        angle: float,
    ) -> np.ndarray:
        """Apply the QUBO pair-term phase to basis states with both bits set."""
        new_state = state.copy()
        for basis_index in range(len(state)):
            bit1_is_one = (basis_index >> qubit1) & 1
            bit2_is_one = (basis_index >> qubit2) & 1
            if bit1_is_one and bit2_is_one:
                new_state[basis_index] *= np.exp(1j * angle)
        return new_state

    def evolve_state(
        self,
        circuit: QuantumCircuit,
        qubo: QUBOFormulation,
    ) -> np.ndarray:
        """Prepare the state produced by all cost-mixer layer pairs."""
        state_size = 2**self.num_qubits
        state = np.ones(state_size, dtype=complex) / np.sqrt(state_size)
        working_circuit = QuantumCircuit(
            num_qubits=circuit.num_qubits,
            depth=circuit.depth,
            parameters=circuit.parameters.copy(),
            mixer_hamiltonian=circuit.mixer_hamiltonian,
            cost_hamiltonian=circuit.cost_hamiltonian,
            quantum_state=state,
        )

        for layer in range(circuit.depth):
            gamma = circuit.parameters[2 * layer]
            beta = circuit.parameters[2 * layer + 1]
            self.apply_cost_hamiltonian(working_circuit, qubo, gamma)
            self.apply_mixer_hamiltonian(working_circuit, beta)

        if working_circuit.quantum_state is None:
            raise RuntimeError("State evolution did not produce a statevector.")
        return working_circuit.quantum_state

    def measure_quantum_state(
        self,
        circuit: QuantumCircuit,
        num_samples: int = 1000,
    ) -> List[List[int]]:
        """Sample binary decisions from the statevector probabilities."""
        if circuit.quantum_state is None:
            raise ValueError("The circuit state has not been initialized.")
        if num_samples < 1:
            raise ValueError("The number of samples must be positive.")

        probabilities = np.abs(circuit.quantum_state) ** 2
        probability_sum = probabilities.sum()
        if not np.isclose(probability_sum, 1.0, atol=1e-10):
            raise ValueError("Statevector probabilities must sum to one.")
        probabilities = probabilities / probability_sum

        sampled_indices = np.random.choice(
            len(probabilities), size=num_samples, p=probabilities
        )
        return [
            [(int(state_index) >> qubit) & 1 for qubit in range(self.num_qubits)]
            for state_index in sampled_indices
        ]

    def calculate_objective(
        self,
        solution: List[int],
        qubo: QUBOFormulation,
    ) -> float:
        """Evaluate the QUBO objective for a binary facility-selection vector."""
        if len(solution) != qubo.num_variables:
            raise ValueError("The solution length must match the number of variables.")
        if any(value not in (0, 1, False, True) for value in solution):
            raise ValueError("Every decision variable must be binary.")

        objective = float(qubo.constant_term)
        for i, selected in enumerate(solution):
            if selected:
                objective += float(qubo.linear_terms[i])

        for i in range(qubo.num_variables):
            for j in range(i + 1, qubo.num_variables):
                if solution[i] and solution[j]:
                    objective += float(qubo.quadratic_terms[i, j])

        return objective

    def optimize_parameters(
        self,
        circuit: QuantumCircuit,
        qubo: QUBOFormulation,
        max_iterations: int = 100,
        samples_per_iteration: int = 100,
    ) -> List[List[float]]:
        """Tune parameters by random perturbation and sampled objective estimates."""
        if max_iterations < 1:
            raise ValueError("The number of iterations must be positive.")
        if samples_per_iteration < 1:
            raise ValueError("The number of samples per iteration must be positive.")

        best_parameters = circuit.parameters.copy()
        best_estimated_energy = float("inf")
        parameter_history: List[List[float]] = []
        self.convergence_history = []

        for _ in range(max_iterations):
            candidate_parameters = [
                (parameter + random.uniform(-0.1, 0.1)) % (2.0 * np.pi)
                for parameter in circuit.parameters
            ]
            candidate_circuit = QuantumCircuit(
                num_qubits=circuit.num_qubits,
                depth=circuit.depth,
                parameters=candidate_parameters,
            )
            candidate_circuit.quantum_state = self.evolve_state(
                candidate_circuit, qubo
            )
            samples = self.measure_quantum_state(
                candidate_circuit,
                num_samples=samples_per_iteration,
            )
            estimated_energy = float(
                np.mean(
                    [self.calculate_objective(sample, qubo) for sample in samples]
                )
            )

            parameter_history.append(candidate_parameters.copy())
            if estimated_energy < best_estimated_energy:
                best_estimated_energy = estimated_energy
                best_parameters = candidate_parameters.copy()

            self.convergence_history.append(best_estimated_energy)
            circuit.parameters = candidate_parameters

        circuit.parameters = best_parameters
        return parameter_history

    def run_qaoa_optimization(
        self,
        depth: int = 3,
        max_iterations: int = 100,
        samples_per_iteration: int = 100,
        final_samples: int = 1000,
    ) -> QuantumResult:
        """Run QUBO construction, parameter search, and final measurement."""
        print(f"Starting statevector QAOA simulation with {self.num_facilities} facilities.")
        qubo = self.formulate_qubo()
        print(f"QUBO formulated with {qubo.num_variables} binary variables.")

        circuit = self.initialize_quantum_circuit(depth)
        print(f"Circuit initialized with depth {depth}.")
        parameter_history = self.optimize_parameters(
            circuit,
            qubo,
            max_iterations=max_iterations,
            samples_per_iteration=samples_per_iteration,
        )
        print("Parameter search completed.")

        circuit.quantum_state = self.evolve_state(circuit, qubo)
        final_decisions = self.measure_quantum_state(
            circuit,
            num_samples=final_samples,
        )
        sampled_energies = [
            self.calculate_objective(decision, qubo)
            for decision in final_decisions
        ]
        best_index = int(np.argmin(sampled_energies))

        metrics = {
            "parameter_count": float(2 * depth),
            "statevector_dimension": float(2**self.num_qubits),
            "final_sample_count": float(final_samples),
            "best_parameter_search_estimate": float(self.convergence_history[-1]),
        }

        return QuantumResult(
            best_solution=final_decisions[best_index],
            best_energy=sampled_energies[best_index],
            convergence_history=self.convergence_history.copy(),
            parameter_history=parameter_history,
            circuit_depth=depth,
            quantum_metrics=metrics,
            sampled_energies=sampled_energies,
        )

    def get_quantum_metrics_summary(
        self,
        result: QuantumResult,
    ) -> Dict[str, Any]:
        """Summarize measurable outputs without implying quantum advantage."""
        return {
            "best_sampled_objective": result.best_energy,
            "selected_facilities": sum(result.best_solution),
            "circuit_depth": result.circuit_depth,
            "parameter_count": int(result.quantum_metrics["parameter_count"]),
            "statevector_dimension": int(
                result.quantum_metrics["statevector_dimension"]
            ),
            "final_sample_count": int(result.quantum_metrics["final_sample_count"]),
        }

    def visualize_quantum_results(self, result: QuantumResult) -> None:
        """Plot optimization history, sampled outcomes, and input locations."""
        fig, ((ax1, ax2), (ax3, ax4)) = plt.subplots(
            2, 2, figsize=(15, 11)
        )

        iterations = np.arange(1, len(result.convergence_history) + 1)
        ax1.plot(iterations, result.convergence_history, color="navy", linewidth=2)
        ax1.set_xlabel("Parameter-search iteration")
        ax1.set_ylabel("Best estimated objective")
        ax1.set_title("Best sampled estimate during parameter search")
        ax1.grid(True, alpha=0.3)

        if result.parameter_history:
            parameter_array = np.asarray(result.parameter_history)
            image = ax2.imshow(
                parameter_array.T,
                aspect="auto",
                cmap="viridis",
                interpolation="nearest",
            )
            ax2.set_xlabel("Parameter-search iteration")
            ax2.set_ylabel("Parameter index")
            ax2.set_title("QAOA parameter values")
            fig.colorbar(image, ax=ax2, label="Parameter value (radians)")

        selected = [
            facility
            for index, facility in enumerate(self.facilities)
            if result.best_solution[index]
        ]
        unselected = [
            facility
            for index, facility in enumerate(self.facilities)
            if not result.best_solution[index]
        ]
        if unselected:
            ax3.scatter(
                [facility.x for facility in unselected],
                [facility.y for facility in unselected],
                marker="x",
                s=75,
                color="gray",
                label="Unselected facility",
            )
        if selected:
            ax3.scatter(
                [facility.x for facility in selected],
                [facility.y for facility in selected],
                marker="o",
                s=90,
                color="tab:blue",
                label="Selected facility",
            )
        if self.demand_zones:
            ax3.scatter(
                [zone.x for zone in self.demand_zones],
                [zone.y for zone in self.demand_zones],
                marker="^",
                s=80,
                color="tab:orange",
                label="Demand zone (not modeled in QUBO)",
            )
        for facility in self.facilities:
            ax3.annotate(
                f"F{facility.id}",
                (facility.x, facility.y),
                xytext=(4, 4),
                textcoords="offset points",
                fontsize=8,
            )
        ax3.set_xlabel("X coordinate (coordinate units)")
        ax3.set_ylabel("Y coordinate (coordinate units)")
        ax3.set_title("Facility decisions and demand-zone locations")
        ax3.grid(True, alpha=0.3)
        ax3.legend(fontsize=8)

        ax4.hist(
            result.sampled_energies,
            bins="auto",
            color="tab:purple",
            alpha=0.75,
            edgecolor="white",
        )
        ax4.set_xlabel("Sampled QUBO objective")
        ax4.set_ylabel("Sample count")
        ax4.set_title("Objective values in the final measurement samples")
        ax4.grid(True, axis="y", alpha=0.3)

        fig.tight_layout()
        plt.show()

## Define and run the example

The inputs below provide eight candidate facilities and four demand-zone locations. Facility coordinates, capacities, fixed costs, demand values, and the illustrative metadata are kept as supplied. Only the facility-selection objective defined above is evaluated; capacity and demand are not constraints in this example.

In [ ]:
facilities = [
    Facility(1, 10, 20, 100, 50000, quantum_amplitude=0.8 + 0.2j),
    Facility(2, 30, 15, 120, 60000, quantum_amplitude=0.7 + 0.3j),
    Facility(3, 25, 35, 90, 45000, quantum_amplitude=0.9 + 0.1j),
    Facility(4, 40, 25, 110, 55000, quantum_amplitude=0.6 + 0.4j),
    Facility(5, 15, 30, 105, 52000, quantum_amplitude=0.85 + 0.15j),
    Facility(6, 35, 20, 95, 48000, quantum_amplitude=0.75 + 0.25j),
    Facility(7, 20, 25, 115, 58000, quantum_amplitude=0.65 + 0.35j),
    Facility(8, 45, 30, 100, 51000, quantum_amplitude=0.95 + 0.05j),
]

demand_zones = [
    DemandZone(1, 20, 25, 80, quantum_priority=1.2),
    DemandZone(2, 35, 20, 100, quantum_priority=1.0),
    DemandZone(3, 25, 30, 90, quantum_priority=1.1),
    DemandZone(4, 40, 35, 85, quantum_priority=0.9),
]

quantum_optimizer = QuantumNetworkOptimizer(facilities, demand_zones)

print("=" * 64)
print("STATEVECTOR QAOA FACILITY-SELECTION EXAMPLE")
print("=" * 64)

quantum_result = quantum_optimizer.run_qaoa_optimization(
    depth=3,
    max_iterations=50,
)

print("\n" + "=" * 64)
print("BEST DECISION OBSERVED IN THE FINAL SAMPLES")
print("=" * 64)

selected_facilities = [
    facility
    for index, facility in enumerate(facilities)
    if quantum_result.best_solution[index]
]

print(f"Binary decision vector: {quantum_result.best_solution}")
print(f"Selected facility count: {len(selected_facilities)}")
for facility in selected_facilities:
    print(
        f"  Facility {facility.id}: location=({facility.x}, {facility.y}), "
        f"fixed cost=${facility.fixed_cost:,.0f}"
    )

print(f"\nSampled QUBO objective: {quantum_result.best_energy:.6f}")
print("This is the best objective among the final measurement samples.")

print("\nSimulation metrics:")
for metric, value in quantum_optimizer.get_quantum_metrics_summary(
    quantum_result
).items():
    print(f"  {metric}: {value}")

qubo = quantum_optimizer.formulate_qubo()
assert len(quantum_result.best_solution) == qubo.num_variables
assert all(value in (0, 1) for value in quantum_result.best_solution)
assert np.isclose(
    quantum_result.best_energy,
    quantum_optimizer.calculate_objective(quantum_result.best_solution, qubo),
)
assert len(quantum_result.parameter_history) == len(
    quantum_result.convergence_history
)
assert np.all(np.diff(quantum_result.convergence_history) <= 1e-12)
print("\nDecision-vector and objective consistency checks passed.")

## Inspect the simulation

The first panel shows the best sampled mean-objective estimate encountered during parameter search; it is an estimate based on finite samples, not a proof of global optimality. The parameter panel displays the candidate angles tested. The location panel distinguishes selected facilities, unselected facilities, and demand-zone locations; demand zones are shown for context, not because coverage is enforced. The final panel summarizes objective values drawn from the final statevector distribution.

In [ ]:
quantum_optimizer.visualize_quantum_results(quantum_result)

## Interpretation, strengths, and limitations

### What the computation establishes

The returned decision is the lowest-objective bit string observed in the final set of samples. Its objective is evaluated using the same linear and pairwise QUBO coefficients used to apply the cost phases. Parameter-search convergence records the best sampled mean objective seen so far; because each estimate uses a finite random sample, the curve is an empirical search trace rather than a deterministic convergence guarantee. Different runs can produce different parameter histories and sampled decisions.

There is no reported classical baseline, exact-optimality certificate, quantum accuracy, or quantum advantage in this notebook. Such quantities cannot be inferred from a statevector simulation and the sampled objective values alone. The number of basis states represented by the statevector is \(2^n\), but representing all these amplitudes is not evidence that a physical quantum device explores or evaluates them at no cost.

### Strengths of the demonstrated approach

- The QUBO coefficients map directly to binary facility decisions and can be checked against the objective function.
- Cost and mixer operations make the role of alternating QAOA layers explicit.
- Measurement converts the statevector distribution into candidate classical decisions that can be evaluated under the original QUBO.
- Parameter search and final sampling are separate, making the distinction between tuning estimates and reported samples visible.

### Limitations and modeling boundaries

- A statevector simulation requires memory proportional to \(2^n\) and is therefore limited to small instances.
- The random-perturbation parameter search is stochastic and does not guarantee a global optimum.
- The current QUBO contains no demand-coverage, capacity, facility-opening, or disruption-scenario constraints. These cannot be inferred from the supplied demand and capacity fields.
- The distance reward is part of a stipulated composite objective and is not a calibrated monetary resilience benefit.
- The example does not compare against a classical optimizer or hardware implementation, so it supports no claim of quantum speedup or solution-quality advantage.

A more complete network-design model would first need explicit definitions of service coverage, capacity feasibility, disruption scenarios, and the costs or penalties associated with them. Those modeling choices are outside the objective encoded here.